# Skin Cancer Detection — ISIC 2024
## Notebook 6: Evaluating Without Accuracy Metrics

Goal: Show that standard accuracy is meaningless on imbalanced data.


In [5]:
import pandas as pd
import numpy as np
import lightgbm as lgbm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report

data = pd.read_csv('/content/encoded_train_metadata.csv')
data = data.drop(columns=['anatom_site_encoded'],errors='ignore')

y = data['target']
X = data.drop(columns=['target'])

print("Data loaded!")
print("Cancer cases:", y.sum())

Data loaded!
Cancer cases: 393


1. Train LightGBM with scale_pos_weight=1019
2. Convert probability predictions to hard labels
3. Generate classification report
4. Show how accuracy hides terrible cancer detection

In [2]:
params = {
    'objective': 'binary',
    'metric':'average_precision',
    'scale_pos_weight':1019,
    'n_estimators':1000,
    'learning_rate':0.05,
    'num_leaves':31,
    'random_state':42,
    'n_jobs':-1,
    'verbose':-1
}



In [3]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_prediction = np.zeros(len(X))

fold = 1

for train_idx,val_idx in skf.split(X,y):

  X_train, y_train = X.iloc[train_idx],y.iloc[train_idx]
  X_val,y_val = X.iloc[val_idx],y.iloc[val_idx]

  model = lgbm.LGBMClassifier(**params)
  model.fit(X_train,y_train)

  val_pred = model.predict_proba(X_val)[:,1]
  oof_prediction[val_idx] = val_pred

  print(f"Fold {fold} complete")
  fold += 1

print("All folds trained!")

Fold 1 complete
Fold 2 complete
Fold 3 complete
Fold 4 complete
Fold 5 complete
All folds trained!


## Step 2 — Convert Probabilities to Hard Labels

Our model outputs **probabilities** between 0 and 1.
To generate a classification report, we need
**hard labels** (0 or 1).

In [4]:
y_pred = (oof_prediction >= 0.5).astype(int)
print("Predicted cancer cases:", y_pred.sum())
print("Actual cancer cases:", y.sum())

Predicted cancer cases: 29352
Actual cancer cases: 393


## Step 3 — Classification Report

In [6]:
print(classification_report(y, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.93      0.96    400666
           1       0.01      0.54      0.01       393

    accuracy                           0.93    401059
   macro avg       0.50      0.73      0.49    401059
weighted avg       1.00      0.93      0.96    401059



## What Does This Tell Us?

Look at the accuracy — it is very high (~99.9%).
But look at the Recall for class 1 (cancer).
The model is missing most cancer cases.
High accuracy is hiding terrible cancer detection.
This is why we use PR-AUC instead.